# P00b — build the panel from a folder (the path in the FIRST cell, every step's result kept where you can print it)

Set `INPUT_DIR` below to the parent folder of your exports and run the cells top to bottom. Each step runs in sequence and leaves what it
produced on the object `B` (`B.input_audit`, `B.stats`, `B.validity`, `B.precision_report`, `B.timing_table`, ... — the last cell lists them all).
Print any of them between the steps when you want to look; otherwise just keep running. The output folder is `<INPUT_DIR>/output` unless you
set `OUTPUT_DIR`. Nothing of the DESIGN is chosen here — every model sets its own in its CELL 1. The same steps from a terminal:
`python build_panel.py --input <folder>`; the R twin: `Rscript build_panel.R input=<folder>`.


In [ ]:
# ---- CELL 1: the folder, and the few options of the build (everything else comes from P00_Settings' defaults) ----
import os
INPUT_DIR    = os.environ.get("REWARD_INPUT_DIR", r"D:\LKT\RWD_Artal\data")   # the PARENT folder of your exports (any depth below it is read)
OUTPUT_DIR   = os.environ.get("REWARD_OUTPUT_DIR") or None   # None = <INPUT_DIR>\output (the P00 layout) | or a folder of your choice
WORKERS      = None        # PASS A worker processes: None = every core the platform allows (Windows: at most 60)
FORCE_REBUILD = False      # False = a valid panel already in OUTPUT_DIR is kept (said); True = build it again
FUND_PATH    = None        # the fund-release workbook: None = the path in _paths.py / P00_Settings
CROSSWALK_PATH = None      # the sub-watershed crosswalk workbook: None = the path in _paths.py
RUN_READINESS = True       # the per-variable estimator files, baseline means, the outcome screen and the model-readiness table (step 9)
OUTCOMES     = None        # None = every outcome the panel has | or a list, e.g. ["NDVI", "LAI"]
print(f"[INFO]    exports under {INPUT_DIR} -> output {OUTPUT_DIR or os.path.join(INPUT_DIR, 'output')}")


In [ ]:
# ---- CELL 2: find the engine folder (DIDRDP_ALLRunDID_v20, wherever this notebook was opened from) and start the build object ----
import os, sys, glob, importlib
def _find_engine():
    starts = [os.getcwd(), os.path.dirname(os.path.abspath(globals().get("__vsc_ipynb_file__", os.getcwd())))]
    for _d in starts:
        for _ in range(6):
            cands = [_d] + [p for p in glob.glob(os.path.join(_d, "*")) if os.path.isdir(p)] + [p for p in glob.glob(os.path.join(_d, "*", "*")) if os.path.isdir(p)]
            for c in cands:
                if os.path.exists(os.path.join(c, "_prep_common.py")) and os.path.exists(os.path.join(c, "build_panel.py")): return c
            if os.path.dirname(_d) == _d: break
            _d = os.path.dirname(_d)
    raise RuntimeError("the engine folder (with _prep_common.py and build_panel.py) was not found above this notebook -- open Jupyter from the DIDRDP_ALLRunDID_v20 folder or its parent")
ENGINE = _find_engine()
if ENGINE not in sys.path: sys.path.insert(0, ENGINE)
import build_panel; importlib.reload(build_panel)
B = build_panel.PanelBuild(INPUT_DIR, output_dir=OUTPUT_DIR, workers=WORKERS, force=FORCE_REBUILD, fund=FUND_PATH, crosswalk=CROSSWALK_PATH, readiness=RUN_READINESS, outcomes=OUTCOMES)
print(f"[OK]      engine {ENGINE} | panel will be {B.panel_path}")


## Step 2 — the input audit
Every export file: its `Treat` flag (1 = post / 0 = pre), its `buff_km` codes (0 = treatment, 1–5 = control), its sub-watershed. Kept: `B.files`, `B.site_tagging`, `B.input_audit` (after PASS A).

In [ ]:
B.audit()
print(f"{len(B.files)} export files")   # B.site_tagging / B.input_audit are DataFrames: B.site_tagging.head()


## Step 3 — PASS A
Ingest, harmonise, stable pixel ids, QC, per-file dedup on every core. Kept: `B.registry`, `B.n_pixels_pass_a`, `B.unresolved_cols`, `B.parse_errors`, `B.dedup_conflicts`, `B.shard_paths`, `B.input_audit`. A valid panel already there is kept (`B.panel_kept`).

In [ ]:
B.pass_a()
if not B.panel_kept: print(f"{B.n_pixels_pass_a:,} pixels | {len(B.shard_paths)} year-season blocks | {len(B.parse_errors)} file notes | {len(B.unresolved_cols)} unresolved column notes")


## Step 4 — the fund timing and the dose
Kept: `B.fund`, `B.dose_table`, `B.crosswalk`, `B.fund_tables`.

In [ ]:
B.dose()
B.dose_table.head() if B.dose_table is not None else print("no fund file: the panel carries NaN dose columns")


## Step 5 — PASS B
One ordered panel: cross-file dedup, the design columns (`treat`, `control`, `pre`, `post`, `did`) from the exports' flag, the dose merge. Kept: `B.final_path`, `B.manifest`.

In [ ]:
B.pass_b()
print(B.final_path)


## Step 6 — integrity, validity, duplicates confirmed
Kept: `B.stats` (rows, pixels, cells, unique key values), `B.validity` (`B.validity['VERDICT']`, failures, warnings), `B.readiness_report`, `B.duplicates`.

In [ ]:
B.integrity()
print(B.validity["VERDICT"]); print({k: B.stats[k] for k in ("n_rows", "n_treated_pixels_est", "n_control_pixels_est", "cells_treat_x_post", "duplicates_total")})


## Step 7 — the precision the panel holds, and the outcome identities
Kept: `B.precision_report` (per variable: stored dtype, distinct values, the smallest difference present, the decimals needed, whether float32 would have lost it), `B.identities`.

In [ ]:
B.precision()
B.precision_report


## Step 8 — the season report and the default scenario the models inherit
Kept: `B.season_report`, `B.design` (DESIGN IN EFFECT of the defaults), `B.fragments`, `B.timing_table`, `B.scenario`.

In [ ]:
B.design_defaults()
B.timing_table


## Step 9 — estimator files, baseline means, the outcome screen, model readiness
Kept: `B.baselines`, `B.screen`, `B.readiness_table`. Skipped when `RUN_READINESS = False`.

In [ ]:
B.readiness()
B.readiness_table


## Everything the build left for you
`B.summary()` lists every attribute with what it is; print any of them, e.g. `B.input_audit`, `B.stats['unique_key_values']['Year']`, `B.validity['warnings']`, `B.precision_report[B.precision_report.variable == 'NDVI']`, `B.timings`.

In [ ]:
B.summary()


---
**Or in one go:** `B = build_panel.PanelBuild(INPUT_DIR).run_all()` runs every step in sequence; the attributes above are there afterwards all the same. Next: any model notebook (its CELL 1 sets the design), or `python orchestrator.py --config config/analysis_config.yaml`.